In [1]:
# NativeNext management setup; original course cells follow unchanged.
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T162327-8f0cb35f')
import next_identity as _identity
_identity.write_child_runtime_identity()
import next_adapter as _next
_next.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T162327-8f0cb35f\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261019T162327-8f0cb35f','04_fastai_part2/13_resnet.ipynb')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/13_resnet.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '13_resnet.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hi\20261019T162327-8f0cb35f\source\HaiLong_Colab_Exercises\04_fastai_part2
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hi\20261019T162327-8f0cb35f\source\HaiLong_Colab_Exercises\04_fastai_part2\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
MAX_BATCH_SIZE = 32  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

In [4]:
#|default_exp resnet

# ResNets

In [5]:
#|export
import pickle,gzip,math,os,time,shutil,torch,matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
import fastcore.all as fc
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,nn,optim
from torch.utils.data import DataLoader,default_collate
from torch.nn import init
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *

In [6]:
from fastcore.test import test_close

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'

import logging
logging.disable(logging.WARNING)

set_seed(42)

In [7]:
xl,yl = 'image','label'
name = "zalando-datasets/fashion_mnist"
bs = min(1024, MAX_BATCH_SIZE)
xmean,xstd = 0.28, 0.35

@inplace
def transformi(b): b[xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[xl]]

dsd = load_dataset(name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=4)

README.md: 0.00B [00:00, ?B/s]

C:\Users\[LOCAL_USER]\.v\hlp213\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hi\20261019T162327-8f0cb35f\private_cache\hf\hub\datasets--zalando-datasets--fashion_mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/30.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
#|export
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

In [9]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(fc.risinstance(GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
iw = partial(init_weights, leaky=0.1)

## Going deeper

In [10]:
def get_model(act=nn.ReLU, nfs=(8,16,32,64,128), norm=nn.BatchNorm2d):
    layers = [conv(1, 8, stride=1, act=act, norm=norm)]
    layers += [conv(nfs[i], nfs[i+1], act=act, norm=norm) for i in range(len(nfs)-1)]
    return nn.Sequential(*layers, conv(nfs[-1], 10, act=None, norm=norm, bias=True), nn.Flatten()).to(def_device)

In [11]:
set_seed(42)
lr,epochs = 6e-2,5
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [12]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.039 GB; runtime free 524.826 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 524.826 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.828,0.533,0,train
0.865,0.393,0,eval
0.854,0.416,1,train
0.848,0.428,1,eval
0.870,0.369,2,train
0.865,0.363,2,eval
0.893,0.305,3,train
0.909,0.253,3,eval
0.921,0.222,4,train
0.919,0.223,4,eval


Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 524.426 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 524.323 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.003 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 523.452 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.003 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 523.451 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.003 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 523.886 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.003 GB; configured budget 80.000 GB; additional estimate 0.010 GB; runtime free 523.886 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.003 GB; configured budget 80.000 GB; additional estimate 0.009 GB; runtime free 523.886 GB. Budget is not Drive quota.


## Skip Connections

The ResNet (*residual network*) was introduced in 2015 by Kaiming He et al in the article ["Deep Residual Learning for Image Recognition"](https://arxiv.org/abs/1512.03385). The key idea is using a *skip connection* to allow deeper networks to train successfully.

<img width="700" src="images/att_00042.png">

In [13]:
#|export
def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    return nn.Sequential(conv(ni, nf, stride=1, act=act, norm=norm, ks=ks),
                         conv(nf, nf, stride=stride, act=None, norm=norm, ks=ks))

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        self.idconv = fc.noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None)
        self.pool = fc.noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)
        self.act = act()

    def forward(self, x): return self.act(self.convs(x) + self.idconv(self.pool(x)))

**Post-lesson update**: Piotr Czapla noticed that we forgot to include `norm=norm` in the call to `_conv_block` above, so the resnets in the lesson didn't have batchnorm in the resblocks! After fixing this, we discovered that initializing the `conv2` batchnorm weights to zero makes things worse in every model we tried, so we removed that. That init method was originally introduced to handle training extremely deep models (much deeper than we use here) -- it appears from this little test that it might be worse for less deep models.

In [14]:
def get_model(act=nn.ReLU, nfs=(8,16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 8, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [nn.Flatten(), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers).to(def_device)

In [15]:
def _print_shape(hook, mod, inp, outp): print(type(mod).__name__, inp[0].shape, outp.shape)
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, cbs=[DeviceCB(), SingleBatchCB()])
with Hooks(model, _print_shape) as hooks: learn.fit(1, train=False)

ResBlock torch.Size([64, 1, 28, 28]) torch.Size([64, 8, 28, 28])
ResBlock torch.Size([64, 8, 28, 28]) torch.Size([64, 16, 14, 14])
ResBlock torch.Size([64, 16, 14, 14]) torch.Size([64, 32, 7, 7])
ResBlock torch.Size([64, 32, 7, 7]) torch.Size([64, 64, 4, 4])
ResBlock torch.Size([64, 64, 4, 4]) torch.Size([64, 128, 2, 2])
ResBlock torch.Size([64, 128, 2, 2]) torch.Size([64, 256, 1, 1])
Flatten torch.Size([64, 256, 1, 1]) torch.Size([64, 256])
Linear torch.Size([64, 256]) torch.Size([64, 10])
BatchNorm1d torch.Size([64, 10]) torch.Size([64, 10])


In [16]:
@fc.patch
def summary(self:Learner):
    res = '|Module|Input|Output|Num params|\n|--|--|--|--|\n'
    tot = 0
    def _f(hook, mod, inp, outp):
        nonlocal res,tot
        nparms = sum(o.numel() for o in mod.parameters())
        tot += nparms
        res += f'|{type(mod).__name__}|{tuple(inp[0].shape)}|{tuple(outp.shape)}|{nparms}|\n'
    with Hooks(self.model, _f) as hooks: self.fit(1, lr=1, train=False, cbs=SingleBatchCB())
    print("Tot params: ", tot)
    if fc.IN_NOTEBOOK:
        from IPython.display import Markdown
        return Markdown(res)
    else: print(res)

In [17]:
TrainLearner(get_model(), dls, F.cross_entropy, cbs=DeviceCB()).summary()

Tot params:  1228908


|Module|Input|Output|Num params|
|--|--|--|--|
|ResBlock|(64, 1, 28, 28)|(64, 8, 28, 28)|712|
|ResBlock|(64, 8, 28, 28)|(64, 16, 14, 14)|3696|
|ResBlock|(64, 16, 14, 14)|(64, 32, 7, 7)|14560|
|ResBlock|(64, 32, 7, 7)|(64, 64, 4, 4)|57792|
|ResBlock|(64, 64, 4, 4)|(64, 128, 2, 2)|230272|
|ResBlock|(64, 128, 2, 2)|(64, 256, 1, 1)|919296|
|Flatten|(64, 256, 1, 1)|(64, 256)|0|
|Linear|(64, 256)|(64, 10)|2560|
|BatchNorm1d|(64, 10)|(64, 10)|20|


In [18]:
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
MomentumLearner(model, dls, F.cross_entropy, cbs=DeviceCB()).lr_find()

In [19]:
lr = 2e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [20]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.093 GB; runtime free 523.843 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.001 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 523.843 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.830,0.570,0,train
0.819,0.518,0,eval
0.866,0.396,1,train
0.863,0.380,1,eval
0.886,0.325,2,train
0.885,0.334,2,eval
0.910,0.256,3,train
0.916,0.230,3,eval
0.937,0.179,4,train
0.930,0.199,4,eval


Storage check (checkpoint_write): artifacts 0.006 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 523.358 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.016 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 523.393 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.030 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 522.506 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.030 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 522.506 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.031 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 522.378 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.031 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 522.378 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.031 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 522.378 GB. Budget is not Drive quota.


In [21]:
import timm
from timm.models.resnet import BasicBlock, ResNet, Bottleneck

In [22]:
' '.join(timm.list_models('*resnet*'))

'cspresnet50 cspresnet50d cspresnet50w eca_resnet33ts ecaresnet26t ecaresnet50d ecaresnet50d_pruned ecaresnet50t ecaresnet101d ecaresnet101d_pruned ecaresnet200d ecaresnet269d ecaresnetlight gcresnet33ts gcresnet50t inception_resnet_v2 lambda_resnet26rpt_256 lambda_resnet26t lambda_resnet50ts legacy_seresnet18 legacy_seresnet34 legacy_seresnet50 legacy_seresnet101 legacy_seresnet152 nf_ecaresnet26 nf_ecaresnet50 nf_ecaresnet101 nf_resnet26 nf_resnet50 nf_resnet101 nf_seresnet26 nf_seresnet50 nf_seresnet101 resnet10t resnet14t resnet18 resnet18d resnet26 resnet26d resnet26t resnet32ts resnet33ts resnet34 resnet34d resnet50 resnet50_clip resnet50_clip_gap resnet50_gn resnet50_mlp resnet50c resnet50d resnet50s resnet50t resnet50x4_clip resnet50x4_clip_gap resnet50x16_clip resnet50x16_clip_gap resnet50x64_clip resnet50x64_clip_gap resnet51q resnet61q resnet101 resnet101_clip resnet101_clip_gap resnet101c resnet101d resnet101s resnet152 resnet152c resnet152d resnet152s resnet200 resnet200d 

```
resnet18:  block=BasicBlock, layers=[2, 2, 2, 2]
resnet18d: block=BasicBlock, layers=[2, 2, 2, 2], stem_width=32, stem_type='deep', avg_down=True
resnet10t: block=BasicBlock, layers=[1, 1, 1, 1], stem_width=32, stem_type='deep_tiered', avg_down=True
```

In [23]:
model = timm.create_model('resnet18d', in_chans=1, num_classes=10)
# model = ResNet(in_chans=1, block=BasicBlock, layers=[2,2,2,2], stem_width=32, avg_down=True)

In [24]:
lr = 2e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [25]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.006 GB; configured budget 80.000 GB; additional estimate 0.571 GB; runtime free 522.277 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.006 GB; configured budget 80.000 GB; additional estimate 0.053 GB; runtime free 522.277 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.823,0.494,0,train
0.826,0.505,0,eval
0.858,0.396,1,train
0.877,0.340,1,eval
0.884,0.315,2,train
0.887,0.305,2,eval
0.910,0.245,3,train
0.911,0.234,3,eval
0.936,0.170,4,train
0.923,0.213,4,eval


Storage check (checkpoint_write): artifacts 0.051 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 522.229 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.140 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 522.140 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.275 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 521.867 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.275 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 521.867 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.275 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 521.740 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.275 GB; configured budget 80.000 GB; additional estimate 0.143 GB; runtime free 521.740 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.275 GB; configured budget 80.000 GB; additional estimate 0.053 GB; runtime free 521.740 GB. Budget is not Drive quota.


## Export -

In [26]:
# The bundled miniai package is ready to import; no export is required.

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [27]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [28]:
# NativeNext independent numerical probe.
_next.probe(globals())


__HAILONG_NATIVE_NEXT_FULL_EVIDENCE__{"contract": {"passed": true, "target": "04_fastai_part2/13_resnet.ipynb", "ordinary_fits": 3, "ordinary_epochs": 15, "ordinary_train_batches": 28125, "ordinary_valid_batches": 2355, "evaluation_single_batch_demos": 2, "lr_finder_demos": 1}, "numerical_artifact_subprobe_only": true, "numerical_models": [{"cell_index": 14, "original_epochs": 5, "architecture": {"kind": "original_six_convolution_baseline", "convolutions": 6}, "probe_rows": 64, "cross_entropy": 0.2269546389579773, "accuracy": 0.90625, "model_checkpoint_snapshot_equal": true, "exact_saved_inference_equal": true, "activation_statistics_retained": true, "optimizer_strict_roundtrip": true, "preserved_gradients_restored": true, "rng_validation": {"python_numpy_torch_cpu_generator_restore": true, "cuda_state_devices": 1, "cuda_validation": "saved ByteTensor structure only; no GPU generator replay claim"}}, {"cell_index": 26, "original_epochs": 5, "architecture": {"kind": "original_custom_res

{'contract': {'passed': True,
  'target': '04_fastai_part2/13_resnet.ipynb',
  'ordinary_fits': 3,
  'ordinary_epochs': 15,
  'ordinary_train_batches': 28125,
  'ordinary_valid_batches': 2355,
  'evaluation_single_batch_demos': 2,
  'lr_finder_demos': 1},
 'numerical_artifact_subprobe_only': True,
 'numerical_models': [{'cell_index': 14,
   'original_epochs': 5,
   'architecture': {'kind': 'original_six_convolution_baseline',
    'convolutions': 6},
   'probe_rows': 64,
   'cross_entropy': 0.2269546389579773,
   'accuracy': 0.90625,
   'model_checkpoint_snapshot_equal': True,
   'exact_saved_inference_equal': True,
   'activation_statistics_retained': True,
   'optimizer_strict_roundtrip': True,
   'preserved_gradients_restored': True,
   'rng_validation': {'python_numpy_torch_cpu_generator_restore': True,
    'cuda_state_devices': 1,
    'cuda_validation': 'saved ByteTensor structure only; no GPU generator replay claim'}},
  {'cell_index': 26,
   'original_epochs': 5,
   'architecture